# Train the apa_camera rat detector

Downloads a labelled dataset from Roboflow and trains a single-class "rat" detector, sized for later Hailo-8L conversion (`tools/convert_to_hailo.py`, repo root).

Run this notebook from **this folder** (`src/modules/variants/apa_camera/training/`) so the `download_roboflow_dataset` / `train` module imports and `requirements.txt` below resolve. Needs a GPU for a real training run -- Colab (with a GPU runtime) or a local GPU workstation both work; CPU works too but will be slow for the full epoch count. On Colab, clone or upload the repo first so this folder (and `requirements.txt`) actually exists in the runtime's filesystem, then `%cd` into it before running the cells below.

Full walkthrough (labelling conventions, hardware notes, deployment): `docs/readthedocs/apa_rat_detector_training.md`.

## 1. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 2. Roboflow API key

Use your **Private API Key** (Roboflow → Workspace Settings → API Keys), not the Publishable one -- the Publishable key is for embedding in client-side hosted-inference widgets and can't pull a full dataset export (images + annotations). The Private key is the one with download access.

This prompts with `getpass` so the key is typed, not stored in the notebook file -- don't paste it into a code cell's source directly, and don't commit a run of this notebook that has the key printed in an output cell.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ROBOFLOW_API_KEY"):
    os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow Private API Key: ")

## 3. Point at the Roboflow project

Slugs come from the project URL: `app.roboflow.com/<workspace>/<project>/...`

In [ ]:
from download_roboflow_dataset import get_project, list_versions, download_dataset

ROBOFLOW_WORKSPACE = "sidb-workshop"
ROBOFLOW_PROJECT = "rat-tracker-zh4ex"

project = get_project(ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT)
list_versions(project)

## 4. Download the dataset

Set `ROBOFLOW_VERSION` to one of the numbers printed above.

In [ ]:
from pathlib import Path

ROBOFLOW_VERSION = 1  # <-- set this from the list printed above

data_yaml = download_dataset(project, ROBOFLOW_VERSION, fmt="yolov11", out=Path("training_data"))
data_yaml

## 5. Train

`imgsz` must match whatever you'll later pass to `tools/convert_to_hailo.py --imgsz` -- leave it at 640 unless you have a specific reason to change it.

In [ ]:
from train import train_model

ratnet_path = train_model(
    data=data_yaml,
    base="yolo11n.pt",
    epochs=150,
    imgsz=640,
    batch=16,
    device=None,  # e.g. "0" for first GPU, "cpu" to force CPU
)
ratnet_path

## 6. Next: convert + deploy

Conversion to a Hailo `.hef` needs **x86-64 Linux** with the Hailo Dataflow Compiler installed (not ARM, so not on the Pi, and not inside this notebook if you're running on Colab or a non-Linux workstation) -- run this as a separate step on a machine that has it:

```bash
pip install "ultralytics>=8.3" hailo_dataflow_compiler onnx onnxruntime
python tools/convert_to_hailo.py --model <path to ratnet.pt printed above> \
    --calib-dir /path/to/64-128/representative/arena/images
```

Then copy the resulting `.hef` to the Pi and set `object_detection.model_path` in the module config. See `docs/readthedocs/apa_rat_detector_training.md` (Steps 5-6) for the full detail, including verifying against the live MJPEG preview before trusting it in a real session.